# Radon post-detect raster lab

Reads a **manual label file**, pulls each label's **exact vectors** (via
`LabelEntry.vector_signatures` -> `label_schema.path_signature`), then renders each label's
vector cluster to a **raster** the same way `VectorClassification/parse.py` renders it right
before handing it to PaddleOCR's recognizer -- same dpi/padding rule
(`ocr_prep.render_cluster_with_dynamic_dpi`, `VectorClassification/config.py`'s own
`OCR_DPI`/`MIN_RENDER_SIDE_PX`/`MAX_RENDER_DPI`/`RENDER_PADDING_EXTRA_PT`). Each sample is
therefore a stand-in for "the crop as it looks right after PaddleOCR's detector has localized
it" -- a raster, not vector geometry.

Seven independent, tentative/experimental rotation-angle estimators are then read off each
sample: (1) a **Radon sparseness read** -- the variance of each theta's projection column
(Postl's method) over a real `skimage.transform.radon` sinogram, giving the sparsest theta,
the densest theta, and the theta whose 90-degree-apart partner differs from it the most in
variance; (2) a **Hough line transform** over an Otsu-thresholded ink mask, taking the single
strongest detected line's angle; (3) **`cv2.minAreaRect`** over the same ink mask; (4) a
**RANSAC line fit** over the same mask's ink-pixel coordinates; and (5) an **FFT power-spectrum**
read of the raster's dominant frequency-domain orientation. Each of the seven resulting angles
is compared against the label's own `expected_rotation` (a signed, mod-180
`angle_diff_mod180`), printed per sample, and finally aggregated into a mean-absolute-error bar
chart across all seven approaches -- still experimental/exploratory, not yet integrated back
into the pipeline as a real deskew step.

## 0 - Config

In [ ]:
from pathlib import Path

import numpy as np

LABEL_PATH        = None     # None -> first (name-sorted) outputs/labels/*.json
PDF_PATH_OVERRIDE = None     # None -> LabelSet.pdf_path (resolved vs repo root)
PAGE_INDEX        = None     # None -> every page present in the label file

# A real Radon sinogram's projections repeat every 180 degrees, so this is a
# plain, independent full sweep -- not paired with a perpendicular companion
# the way a pure-vector deskew sweep would be.
SINOGRAM_THETA = np.arange(0.0, 180.0, 1.0)

In [ ]:
def angle_diff_mod180(detected_deg, expected_deg):
    """Signed smallest difference in [-90, 90) between two angles, treated as
    undirected (mod-180) line orientations rather than directed vectors."""
    return ((detected_deg - expected_deg + 90.0) % 180.0) - 90.0

## 1 - Path bootstrap

In [ ]:
import os, sys
_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.append(_root)

## 2 - Labels -> the exact labelled vector set per label

`extract_vectors(page)` once per page (raw `get_drawings()` geometry -- no classification,
no clustering, no rendering), keyed by `path_signature`; each label pulls its own vectors by
the signatures it stored.

In [ ]:
from rastervec.P1_Reading_Native.reader import Reader
from rastervec.P1_Reading_Native.vector_extract import extract_vectors
from rastervec.Evaluation.Labelling.label_schema import (
    load_labels, split_labelset_by_source, path_signature)
from rastervec.commons.helpers.geometry import union_bbox
from rastervec.commons.paths import output_dir, REPO_ROOT


def _resolve_pdf(raw):
    p = Path(str(raw).replace(chr(92), "/"))
    if p.is_file():
        return p
    for base in (Path.cwd(), REPO_ROOT):
        for cand in ((base / p), (base / p.name)):
            if cand.is_file():
                return cand.resolve()
    for sub in ("references", "references2"):
        cand = REPO_ROOT / sub / p.name
        if cand.is_file():
            return cand
    raise FileNotFoundError(f"cannot locate PDF {raw!r} (cwd={Path.cwd()}, repo={REPO_ROOT})")


def _pick_label_path():
    if LABEL_PATH:
        return Path(LABEL_PATH)
    cands = sorted(Path(output_dir("labels")).glob("*.json"))
    if not cands:
        raise FileNotFoundError("no outputs/labels/*.json; set LABEL_PATH")
    return cands[0]


label_path = _pick_label_path()
labels = load_labels(label_path)
pdf_path = _resolve_pdf(PDF_PATH_OVERRIDE or labels.pdf_path)
manual = split_labelset_by_source(labels)["manual"].entries
print(f"label file: {label_path}")
print(f"pdf:        {pdf_path}")
print(f"manual entries: {len(manual)}")

by_page = {}
for e in manual:
    if PAGE_INDEX is None or e.page_index == PAGE_INDEX:
        by_page.setdefault(e.page_index, []).append(e)

matched = []          # (entry, [Vector, ...])
with Reader(pdf_path) as r:
    for pidx in sorted(by_page):
        page_vectors = extract_vectors(r.get_page(pidx))
        sigmap = {path_signature(v): v for v in page_vectors}
        for e in by_page[pidx]:
            sel = [sigmap[s] for s in e.vector_signatures if s in sigmap]
            matched.append((e, sel))
            print(f"  p{pidx}  {e.text!r:12}  rot={e.expected_rotation:>4}  "
                  f"found {len(sel)}/{len(e.vector_signatures)} vectors")

print(f"\nmatched clusters: {len(matched)}")

## 3 - Render each sample the way OCR sees it (pre-recognition render + pad)

Same render/pad path `VectorClassification/parse.py` uses right before calling PaddleOCR's
detector/recognizer: `ocr_prep.render_cluster_with_dynamic_dpi` at `OCR_DPI`, dpi bumped up
(never down) for a small cluster, padded by half the cluster's own max stroke width plus
`RENDER_PADDING_EXTRA_PT` (`parse.py::_cluster_render_padding`).

In [ ]:
from rastervec.commons.renderer import ocr_prep
from rastervec.P3_Vector_Parsing.VectorClassification.config import (
    OCR_DPI, MIN_RENDER_SIDE_PX, MAX_RENDER_DPI, RENDER_PADDING_EXTRA_PT)


def _cluster_render_padding(vectors):
    """Mirrors `VectorClassification/parse.py::_cluster_render_padding`."""
    return max((v.width or 0.0) for v in vectors) / 2.0 + RENDER_PADDING_EXTRA_PT


samples = []   # dict(entry, vectors, image, gray, dpi_used, padding)
for e, vec in matched:
    if not vec:
        print(f"  {e.text!r:14} -- no vectors, skipped")
        continue
    padding = _cluster_render_padding(vec)
    image, dpi_used = ocr_prep.render_cluster_with_dynamic_dpi(
        vec, OCR_DPI, MIN_RENDER_SIDE_PX, MAX_RENDER_DPI, padding)
    gray = np.asarray(image.convert("L"), dtype=float)
    samples.append(dict(entry=e, vectors=vec, image=image, gray=gray,
                        dpi_used=dpi_used, padding=padding))
    print(f"  {e.text!r:14} rot={e.expected_rotation:>4}  "
          f"{len(vec)} vectors  render {gray.shape[1]}x{gray.shape[0]}px @ {dpi_used}dpi")

print(f"\nsamples: {len(samples)}")

## 4 - Sinogram (tentative, experimental)

Just a raw `skimage.transform.radon` call over each sample's raster -- no analysis of the
result yet, purely to look at what the transform produces on real post-detect-style crops.

In [ ]:
from skimage.transform import radon

for s in samples:
    s["sinogram"] = radon(s["gray"], theta=SINOGRAM_THETA, circle=False)

## 4a - Sparseness-based rotation from the Radon sinogram (variance / Postl's method)

For each sample, take the variance of each theta column in the sinogram already computed
above (`s["sinogram"]`, one column per `SINOGRAM_THETA` entry). A peaked/spiky projection
scores high variance ("sparsest"); a flat/uniform projection scores low variance ("densest").
Also finds, over 90-degree-apart pairs of thetas (`SINOGRAM_THETA` spans exactly 0..179 at
1-degree steps, so index `i` and `i+90` are always a valid pair), the pair whose variance
differs the most -- reporting the **sparser** member of that pair. Each of the three
resulting thetas is compared against the label's own `expected_rotation` via
`angle_diff_mod180`.

In [ ]:
n_theta = len(SINOGRAM_THETA)
assert n_theta == 180, f"90-degree pairing assumes a full 0..179 sweep, got {n_theta} thetas"

for s in samples:
    variance = s["sinogram"].var(axis=0)
    s["theta_sparsest"] = float(SINOGRAM_THETA[variance.argmax()])
    s["theta_densest"] = float(SINOGRAM_THETA[variance.argmin()])

    half, partner = variance[:90], variance[90:180]
    i = int(np.argmax(np.abs(half - partner)))
    s["theta_most_different"] = float(SINOGRAM_THETA[i if half[i] >= partner[i] else i + 90])
    s["theta_most_different_diff"] = float(abs(half[i] - partner[i]))

    e = s["entry"]
    print(f"  {e.text!r:14} expected={e.expected_rotation:>6.1f}  "
          f"sparsest={s['theta_sparsest']:>5.1f} "
          f"(diff {angle_diff_mod180(s['theta_sparsest'], e.expected_rotation):+6.1f})  "
          f"densest={s['theta_densest']:>5.1f} "
          f"(diff {angle_diff_mod180(s['theta_densest'], e.expected_rotation):+6.1f})  "
          f"most_diff={s['theta_most_different']:>5.1f} "
          f"(diff {angle_diff_mod180(s['theta_most_different'], e.expected_rotation):+6.1f})")

## 5 - Plot: raster crop + sinogram, per sample

In [ ]:
import matplotlib.pyplot as plt

_THETA_MARKER_COLORS = {
    "sparsest": "#2a78d6",
    "densest": "#eb6834",
    "most_different": "#eda100",
}

for i, s in enumerate(samples):
    e = s["entry"]
    fig, (left, right) = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(f"#{i}  {e.text!r}   expected_rotation={e.expected_rotation}"
                 f"   ({len(s['vectors'])} vectors)", fontsize=10)

    left.imshow(s["gray"], cmap="gray")
    left.set_title(f"raster ({s['gray'].shape[1]}x{s['gray'].shape[0]}px @ {s['dpi_used']}dpi)",
                   fontsize=8)
    left.set_xticks([]); left.set_yticks([])

    sino = s["sinogram"]
    right.imshow(sino, cmap="gray", aspect="auto",
                extent=(SINOGRAM_THETA[0], SINOGRAM_THETA[-1], 0, sino.shape[0]))
    right.axvline(s["theta_sparsest"], color=_THETA_MARKER_COLORS["sparsest"],
                  linestyle="--", linewidth=1.5, label=f"sparsest {s['theta_sparsest']:.0f}°")
    right.axvline(s["theta_densest"], color=_THETA_MARKER_COLORS["densest"],
                  linestyle="--", linewidth=1.5, label=f"densest {s['theta_densest']:.0f}°")
    right.axvline(s["theta_most_different"], color=_THETA_MARKER_COLORS["most_different"],
                  linestyle="--", linewidth=1.5,
                  label=f"most-diff-by-90 {s['theta_most_different']:.0f}°")
    right.legend(fontsize=6, loc="upper right")
    right.set_title("sinogram", fontsize=8)
    right.set_xlabel("theta (deg)"); right.set_ylabel("detector position (px)")

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()

## 6 - Hough line transform (ink-threshold rotation estimate)

A second, independent rotation estimate per sample: threshold the same grayscale raster
(Otsu, since `replay_drawing_paths` renders dark ink on a light background) into a boolean
ink mask, then run `skimage.transform.hough_line` over that mask and take the single
strongest peak (`hough_line_peaks`, `num_peaks=1`) as "the" detected line. `hough_line`'s
`theta` is the angle of the line's *normal*, not the line itself, so the reported line angle
is `(degrees(theta_peak) + 90) mod 180`. Same 1-degree resolution as the Radon sweep above,
for comparability.

In [ ]:
from skimage.filters import threshold_otsu
from skimage.transform import hough_line, hough_line_peaks

HOUGH_THETA = np.linspace(-90.0, 90.0, 180, endpoint=False) * np.pi / 180.0  # radians

for s in samples:
    gray = s["gray"]
    otsu = threshold_otsu(gray)
    mask = gray < otsu  # ink (dark) = True

    h, theta_arr, d_arr = hough_line(mask, theta=HOUGH_THETA)
    _, theta_peak, d_peak = hough_line_peaks(h, theta_arr, d_arr, num_peaks=1)

    s["hough_mask"] = mask
    s["hough_accumulator"] = h
    s["hough_theta_arr"] = theta_arr
    s["hough_d_arr"] = d_arr

    e = s["entry"]
    if len(theta_peak) == 0:
        s["hough_theta_peak"] = None
        s["hough_d_peak"] = None
        s["hough_angle"] = None
        print(f"  {e.text!r:14} -- no Hough peak found")
        continue

    s["hough_theta_peak"] = float(theta_peak[0])
    s["hough_d_peak"] = float(d_peak[0])
    s["hough_angle"] = (np.degrees(s["hough_theta_peak"]) + 90.0) % 180.0

    diff = angle_diff_mod180(s["hough_angle"], e.expected_rotation)
    print(f"  {e.text!r:14} expected={e.expected_rotation:>6.1f}  "
          f"hough={s['hough_angle']:>5.1f}  diff={diff:+6.1f}")

## 7 - Plot: raster + detected line + Hough accumulator, per sample

In [ ]:
for i, s in enumerate(samples):
    if s["hough_angle"] is None:
        continue
    e = s["entry"]
    fig, (left, right) = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(f"#{i}  {e.text!r}   expected_rotation={e.expected_rotation}"
                 f"   hough_angle={s['hough_angle']:.1f}", fontsize=10)

    h_img, w_img = s["gray"].shape
    left.imshow(s["gray"], cmap="gray")
    cx, cy = w_img / 2.0, h_img / 2.0
    length = max(w_img, h_img)
    angle_rad = np.radians(s["hough_angle"])
    dx, dy = np.cos(angle_rad) * length, np.sin(angle_rad) * length
    left.plot([cx - dx, cx + dx], [cy - dy, cy + dy], color="#eb6834", linewidth=1.5)
    left.set_xlim(0, w_img); left.set_ylim(h_img, 0)
    left.set_title("raster + detected line", fontsize=8)
    left.set_xticks([]); left.set_yticks([])

    h = s["hough_accumulator"]
    theta_arr, d_arr = s["hough_theta_arr"], s["hough_d_arr"]
    right.imshow(h, cmap="gray", aspect="auto",
                extent=(np.degrees(theta_arr[0]), np.degrees(theta_arr[-1]),
                        d_arr[-1], d_arr[0]))
    right.scatter([np.degrees(s["hough_theta_peak"])], [s["hough_d_peak"]],
                  color="#eb6834", s=25, marker="x")
    right.set_title("Hough accumulator (peak marked)", fontsize=8)
    right.set_xlabel("theta (deg, line-normal)"); right.set_ylabel("rho (px)")

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()

## 9 - cv2.minAreaRect rotation estimate

A third, independent rotation estimate per sample: `cv2.minAreaRect` over the same
Otsu-thresholded ink mask (`s["hough_mask"]`, from the Hough section above) gives the smallest
rotated rectangle enclosing every ink pixel. Its `(w, h), angle` describes one edge pair; since
the presumed text-line direction is the **longer** edge, 90 degrees is added whenever `w < h`
before wrapping into `[0, 180)`.

In [ ]:
import cv2

for s in samples:
    mask = s["hough_mask"]
    points = cv2.findNonZero(mask.astype(np.uint8))
    e = s["entry"]
    if points is None or len(points) < 2:
        s["min_area_rect"] = None
        s["min_area_rect_angle"] = None
        print(f"  {e.text!r:14} -- no ink pixels for minAreaRect")
        continue

    rect = cv2.minAreaRect(points)  # ((cx, cy), (w, h), angle)
    (cx, cy), (w, h), angle = rect
    if w < h:
        angle += 90.0  # report the LONG-edge orientation as the line angle
    angle = float(angle % 180.0)

    s["min_area_rect"] = rect
    s["min_area_rect_angle"] = angle

    diff = angle_diff_mod180(angle, e.expected_rotation)
    print(f"  {e.text!r:14} expected={e.expected_rotation:>6.1f}  "
          f"min_area_rect={angle:>5.1f}  diff={diff:+6.1f}")

In [ ]:
for i, s in enumerate(samples):
    if s["min_area_rect_angle"] is None:
        continue
    e = s["entry"]
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(s["gray"], cmap="gray")

    box = cv2.boxPoints(s["min_area_rect"])
    box = np.vstack([box, box[0]])
    ax.plot(box[:, 0], box[:, 1], color="#e87ba4", linewidth=1.5)

    h_img, w_img = s["gray"].shape
    (cx, cy), _, _ = s["min_area_rect"]
    length = max(w_img, h_img)
    angle_rad = np.radians(s["min_area_rect_angle"])
    dx, dy = np.cos(angle_rad) * length, np.sin(angle_rad) * length
    ax.plot([cx - dx, cx + dx], [cy - dy, cy + dy], color="#e87ba4",
            linewidth=1.0, linestyle="--")

    ax.set_title(f"#{i}  {e.text!r}  expected={e.expected_rotation}  "
                 f"min_area_rect={s['min_area_rect_angle']:.1f}", fontsize=9)
    ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

## 10 - RANSAC line fit rotation estimate

A robust alternative to Hough: fit a line to the ink-pixel coordinates of the same mask
(`s["hough_mask"]`) via `skimage.measure.ransac` + `LineModelND`, which down-weights outlier
pixels (stray strokes, noise) rather than taking a global accumulator peak. Guards for too few
ink pixels or a failed fit by leaving the sample's angle as `None`, matching the Hough
no-peak-found pattern.

In [ ]:
from skimage.measure import LineModelND, ransac

for s in samples:
    mask = s["hough_mask"]
    e = s["entry"]
    yx = np.column_stack(np.nonzero(mask))  # (row, col) = (y, x)

    if len(yx) < 2:
        s["ransac_angle"] = None
        s["ransac_points"] = None
        s["ransac_inliers"] = None
        print(f"  {e.text!r:14} -- no ink pixels for RANSAC")
        continue

    try:
        model, inliers = ransac(yx, LineModelND, min_samples=2,
                                 residual_threshold=1.5, max_trials=300)
    except ValueError:
        model, inliers = None, None

    if model is None:
        s["ransac_angle"] = None
        s["ransac_points"] = None
        s["ransac_inliers"] = None
        print(f"  {e.text!r:14} -- RANSAC found no model")
        continue

    origin, direction = model.params  # both (dy, dx), matching yx's column order
    angle = float(np.degrees(np.arctan2(direction[0], direction[1])) % 180.0)

    s["ransac_angle"] = angle
    s["ransac_points"] = yx
    s["ransac_inliers"] = inliers

    diff = angle_diff_mod180(angle, e.expected_rotation)
    print(f"  {e.text!r:14} expected={e.expected_rotation:>6.1f}  "
          f"ransac={angle:>5.1f}  diff={diff:+6.1f}  "
          f"inliers={int(inliers.sum())}/{len(yx)}")

In [ ]:
for i, s in enumerate(samples):
    if s["ransac_angle"] is None:
        continue
    e = s["entry"]
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(s["gray"], cmap="gray")

    yx, inliers = s["ransac_points"], s["ransac_inliers"]
    ax.scatter(yx[~inliers, 1], yx[~inliers, 0], color="#898781", s=2, alpha=0.4)
    ax.scatter(yx[inliers, 1], yx[inliers, 0], color="#008300", s=3, alpha=0.7)

    h_img, w_img = s["gray"].shape
    cx, cy = w_img / 2.0, h_img / 2.0
    length = max(w_img, h_img)
    angle_rad = np.radians(s["ransac_angle"])
    dx, dy = np.cos(angle_rad) * length, np.sin(angle_rad) * length
    ax.plot([cx - dx, cx + dx], [cy - dy, cy + dy], color="#008300", linewidth=1.5)

    ax.set_title(f"#{i}  {e.text!r}  expected={e.expected_rotation}  "
                 f"ransac={s['ransac_angle']:.1f}", fontsize=9)
    ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

## 11 - FFT power-spectrum rotation estimate

A frequency-domain alternative operating on the grayscale raster directly (not the ink mask):
the 2D power spectrum's energy concentrates along the axis **perpendicular** to periodic
text-line spacing, so the dominant orientation is found via a power-weighted 2nd-moment
(structure-tensor) over frequency coordinates -- the largest eigenvector of the weighted
frequency covariance matrix, with the DC bin zeroed out first. The reported line angle is that
dominant-axis angle **+90 degrees**, the same "normal -> line direction" step used for Hough.
No windowing is applied before the FFT (plain rectangular crop), a known source of spectral
leakage -- left as-is, consistent with this notebook's tentative/experimental framing.

In [ ]:
for s in samples:
    gray = s["gray"]
    e = s["entry"]
    h_img, w_img = gray.shape
    if h_img < 4 or w_img < 4:
        s["fft_angle"] = None
        s["fft_power"] = None
        print(f"  {e.text!r:14} -- crop too small for FFT")
        continue

    spectrum = np.fft.fftshift(np.fft.fft2(gray - gray.mean()))
    power = np.abs(spectrum) ** 2

    cy, cx = h_img / 2.0, w_img / 2.0
    vv, uu = np.mgrid[0:h_img, 0:w_img]
    uu = uu - cx
    vv = vv - cy
    power[int(round(cy)), int(round(cx))] = 0.0  # drop the DC bin

    total = power.sum()
    if total <= 0:
        s["fft_angle"] = None
        s["fft_power"] = power
        print(f"  {e.text!r:14} -- flat spectrum, no FFT orientation")
        continue

    Cuu = (power * uu * uu).sum() / total
    Cvv = (power * vv * vv).sum() / total
    Cuv = (power * uu * vv).sum() / total
    cov = np.array([[Cuu, Cuv], [Cuv, Cvv]])
    eigvals, eigvecs = np.linalg.eigh(cov)
    dom = eigvecs[:, np.argmax(eigvals)]  # (u0, v0), dominant frequency-domain axis

    freq_angle = np.degrees(np.arctan2(dom[1], dom[0])) % 180.0
    fft_angle = float((freq_angle + 90.0) % 180.0)  # text-line angle is perpendicular

    s["fft_power"] = power
    s["fft_angle"] = fft_angle

    diff = angle_diff_mod180(fft_angle, e.expected_rotation)
    print(f"  {e.text!r:14} expected={e.expected_rotation:>6.1f}  "
          f"fft={fft_angle:>5.1f}  diff={diff:+6.1f}")

In [ ]:
for i, s in enumerate(samples):
    if s["fft_angle"] is None:
        continue
    e = s["entry"]
    fig, (left, right) = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(f"#{i}  {e.text!r}  expected={e.expected_rotation}  "
                 f"fft={s['fft_angle']:.1f}", fontsize=10)

    left.imshow(s["gray"], cmap="gray")
    h_img, w_img = s["gray"].shape
    cx, cy = w_img / 2.0, h_img / 2.0
    length = max(w_img, h_img)
    angle_rad = np.radians(s["fft_angle"])
    dx, dy = np.cos(angle_rad) * length, np.sin(angle_rad) * length
    left.plot([cx - dx, cx + dx], [cy - dy, cy + dy], color="#4a3aa7", linewidth=1.5)
    left.set_xlim(0, w_img); left.set_ylim(h_img, 0)
    left.set_title("raster + detected line", fontsize=8)
    left.set_xticks([]); left.set_yticks([])

    right.imshow(np.log1p(s["fft_power"]), cmap="gray")
    right.set_title("log power spectrum", fontsize=8)
    right.set_xticks([]); right.set_yticks([])

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()

## 12 - Accuracy comparison across all 7 approaches

Aggregate the per-sample `angle_diff_mod180` values for all seven rotation estimates
(`hough_angle`, `theta_sparsest`, `theta_densest`, `theta_most_different`,
`min_area_rect_angle`, `ransac_angle`, `fft_angle`) against each label's own
`expected_rotation`, and compare mean absolute error side by side -- lower bars are more
accurate. Individual per-sample absolute errors are overlaid as points so the spread behind
each mean is visible too.

In [ ]:
_APPROACH_COLORS = {
    "hough": "#2a78d6",
    "sparsest": "#eb6834",
    "densest": "#1baf7a",
    "most_different": "#eda100",
    "min_area_rect": "#e87ba4",
    "ransac": "#008300",
    "fft": "#4a3aa7",
}
_APPROACH_LABELS = {
    "hough": "Hough line",
    "sparsest": "Radon sparsest",
    "densest": "Radon densest",
    "most_different": "Radon most-diff-by-90",
    "min_area_rect": "cv2 minAreaRect",
    "ransac": "RANSAC line fit",
    "fft": "FFT power-spectrum",
}

errors = {k: [] for k in _APPROACH_COLORS}
for s in samples:
    expected = s["entry"].expected_rotation
    if s.get("hough_angle") is not None:
        errors["hough"].append(abs(angle_diff_mod180(s["hough_angle"], expected)))
    errors["sparsest"].append(abs(angle_diff_mod180(s["theta_sparsest"], expected)))
    errors["densest"].append(abs(angle_diff_mod180(s["theta_densest"], expected)))
    errors["most_different"].append(abs(angle_diff_mod180(s["theta_most_different"], expected)))
    if s.get("min_area_rect_angle") is not None:
        errors["min_area_rect"].append(abs(angle_diff_mod180(s["min_area_rect_angle"], expected)))
    if s.get("ransac_angle") is not None:
        errors["ransac"].append(abs(angle_diff_mod180(s["ransac_angle"], expected)))
    if s.get("fft_angle") is not None:
        errors["fft"].append(abs(angle_diff_mod180(s["fft_angle"], expected)))

approach_keys = list(_APPROACH_COLORS)
means = [float(np.mean(errors[k])) if errors[k] else float("nan") for k in approach_keys]

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(approach_keys))
ax.bar(x, means, color=[_APPROACH_COLORS[k] for k in approach_keys], width=0.55, zorder=2)

rng = np.random.default_rng(0)
for xi, k in zip(x, approach_keys):
    vals = errors[k]
    if not vals:
        continue
    jitter = rng.uniform(-0.12, 0.12, size=len(vals))
    ax.scatter(xi + jitter, vals, color="#52514e", s=14, alpha=0.6, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels([_APPROACH_LABELS[k] for k in approach_keys], fontsize=8, rotation=12)
ax.set_ylabel("mean |angle_diff_mod180| vs expected_rotation (deg)")
ax.set_title("Rotation-estimate accuracy by approach (lower is better)", fontsize=11)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

for xi, m in zip(x, means):
    if not np.isnan(m):
        ax.annotate(f"{m:.1f}°", (xi, m), textcoords="offset points",
                    xytext=(0, 4), ha="center", fontsize=8, color="#0b0b0b")

plt.tight_layout()
plt.show()

print("n samples per approach:", {k: len(v) for k, v in errors.items()})